# DDRI bike_change full161 정본 기본피처 데이터셋 생성 Run-All 노트북

이 노트북은 `군집별 데이터_전체 스테이션/full_data` 정본 CSV를 입력으로 사용하고, 추가 피처를 붙이지 않은 상태에서 `bike_change` 정제 데이터를 다시 생성합니다.

## 생성 원칙
- 입력은 정본 CSV 그대로 사용
- 피처는 정본 데이터 기본 컬럼만 유지
- 타깃만 `bike_change_raw`, `bike_change_deseasonalized`로 생성
- seasonal mean은 `Train 2023` 기준으로 계산


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

root_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
script_name = "09_ddri_bike_change_full161_base_feature_dataset_builder.py"

script_path = None
for base in root_candidates:
    candidate = (base / "works" / "07_prediction_bike_change" / script_name).resolve()
    if candidate.exists():
        script_path = candidate
        break
    candidate = (base / script_name).resolve()
    if candidate.exists():
        script_path = candidate
        break

if script_path is None:
    found = list(Path.cwd().rglob(script_name))
    if found:
        script_path = found[0].resolve()

if script_path is None:
    raise FileNotFoundError("실행 스크립트를 찾지 못했습니다.")

work_dir = script_path.parent
root_dir = work_dir.parents[1]
source_dir = root_dir / "3조 공유폴더" / "군집별 데이터_전체 스테이션" / "full_data"
train_source = source_dir / "ddri_prediction_long_train_2023_2024.csv"
test_source = source_dir / "ddri_prediction_long_test_2025.csv"

print(f"script_path={script_path}")
print(f"train_source={train_source}")
print(f"test_source={test_source}")
if not train_source.exists() or not test_source.exists():
    raise FileNotFoundError("정본 입력 CSV가 없습니다.")


In [ ]:
env = {}
env.update(**__import__('os').environ)
env.setdefault("PYTHONUNBUFFERED", "1")

t0 = time.perf_counter()
process = subprocess.Popen(
    [sys.executable, "-u", str(script_path)],
    cwd=str(root_dir),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
elapsed = time.perf_counter() - t0
print(f"총 실행시간: {elapsed:,.2f}초")
if return_code != 0:
    raise RuntimeError(f"정제 스크립트 실행 실패: return_code={return_code}")


In [ ]:
output_dir = work_dir / "output" / "data"
train_out = output_dir / "ddri_prediction_bike_change_full161_base_train_2023_2024.csv"
test_out = output_dir / "ddri_prediction_bike_change_full161_base_test_2025.csv"
summary_out = output_dir / "ddri_prediction_bike_change_full161_base_feature_summary.csv"
meta_out = work_dir / "ddri_prediction_bike_change_full161_base_feature_meta.json"

train_df = pd.read_csv(train_out, nrows=5)
test_df = pd.read_csv(test_out, nrows=5)
summary_df = pd.read_csv(summary_out)
meta = json.loads(meta_out.read_text(encoding="utf-8"))

display(Markdown("### train 샘플"))
display(train_df)
display(Markdown("### test 샘플"))
display(test_df)
display(Markdown("### feature summary"))
display(summary_df)
display(Markdown(f"**train_rows_after_target**: `{meta['train_rows_after_target']}`  \n**test_rows_after_target**: `{meta['test_rows_after_target']}`  \n**seasonal_groups_from_train_2023**: `{meta['seasonal_groups_from_train_2023']}`"))
